# HW13 — Токенизация текста, инференс BERT и fine-tuning для классификации

**Датасет:** `20 Newsgroups` (4 категории: comp.graphics, rec.sport.baseball, sci.space, talk.politics.misc)  
**Модель:** `distilbert-base-uncased`  
**Задача:** классификация новостных сообщений по тематике

## 1. Импорты, seed и среда

In [13]:
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

# Работаем офлайн — модели уже в кеше
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"torch: {torch.__version__}")

Device: cpu
torch: 2.8.0


## 2. Данные и первичный анализ

In [14]:
# 4 тематические категории из 20 Newsgroups
CATEGORIES = ["comp.graphics", "rec.sport.baseball", "sci.space", "talk.politics.misc"]

raw_train = fetch_20newsgroups(subset="train", categories=CATEGORIES, random_state=SEED, remove=("headers", "footers", "quotes"))
raw_test = fetch_20newsgroups(subset="test", categories=CATEGORIES, random_state=SEED, remove=("headers", "footers", "quotes"))

label_names = raw_train.target_names
num_labels = len(label_names)

print(f"Классы ({num_labels}): {label_names}")
print(f"Train (full): {len(raw_train.data)}")
print(f"Test:         {len(raw_test.data)}")

Классы (4): ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.misc']
Train (full): 2239
Test:         1490


In [15]:
# Создаём validation из train (15%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    raw_train.data, raw_train.target, test_size=0.15, random_state=SEED, stratify=raw_train.target
)
test_texts = raw_test.data
test_labels = raw_test.target

print(f"Train:      {len(train_texts)}")
print(f"Validation: {len(val_texts)}")
print(f"Test:       {len(test_texts)}")

Train:      1903
Validation: 336
Test:       1490


In [16]:
# Примеры текстов и меток
print("Примеры из train:")
for i in range(5):
    text_preview = train_texts[i].replace("\n", " ").strip()[:150]
    print(f"  [{label_names[train_labels[i]]}] {text_preview}...")
    print()

Примеры из train:
  [comp.graphics] There are several public domain utilities available at your usual archive site that allow 'extraction' of single frames from a .gl file, check in the ...

  [comp.graphics] So they should sue the newspaper I got it from for printing it. The article didn't say anything about copyrights.  Louis...

  [rec.sport.baseball] sandiego and graig nettles...

  [comp.graphics] I don't know about that...I've used Photoshop 2.5 on both a 486dx-50 and a Quadra 950...I'd say they are roughly equal.  If anything the 486 was faste...

  [sci.space] Hi all,      I'm trying to get mailing addresses for the following companies.  Specifically, I need addresses for their personnel offices or like bure...



In [17]:
# Распределение классов в train
counts = Counter(train_labels)
print("Распределение классов в train:")
for label_id in sorted(counts):
    print(f"  {label_names[label_id]}: {counts[label_id]} ({counts[label_id]/len(train_labels)*100:.1f}%)")

Распределение классов в train:
  comp.graphics: 496 (26.1%)
  rec.sport.baseball: 508 (26.7%)
  sci.space: 504 (26.5%)
  talk.politics.misc: 395 (20.8%)


## 3. Токенизация

In [18]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.special_tokens_map}")

HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 4s [Retry 3/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 8s [Retry 4/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 8s [Retry 5/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Tokenizer: DistilBertTokenizerFast
Vocab size: 30522
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


In [19]:
# Разбор токенизации на нескольких примерах
sample_texts = [
    train_texts[0].replace("\n", " ").strip()[:200],
    train_texts[1].replace("\n", " ").strip()[:200],
    train_texts[2].replace("\n", " ").strip()[:200],
    "The space shuttle launched successfully from Cape Canaveral.",
    "The pitcher threw a curveball and struck out the batter in the ninth inning.",
]

for text in sample_texts:
    encoded = tokenizer(text, padding=False, truncation=True, max_length=128)
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
    print(f"Text: {text[:100]}")
    print(f"  Tokens ({len(tokens)}): {tokens[:20]}{'...' if len(tokens) > 20 else ''}")
    print(f"  input_ids: {encoded['input_ids'][:20]}{'...' if len(encoded['input_ids']) > 20 else ''}")
    print(f"  attention_mask: {encoded['attention_mask'][:20]}{'...' if len(encoded['attention_mask']) > 20 else ''}")
    print()

Text: There are several public domain utilities available at your usual archive site that allow 'extractio
  Tokens (43): ['[CLS]', 'there', 'are', 'several', 'public', 'domain', 'utilities', 'available', 'at', 'your', 'usual', 'archive', 'site', 'that', 'allow', "'", 'extraction', "'", 'of', 'single']...
  input_ids: [101, 2045, 2024, 2195, 2270, 5884, 16548, 2800, 2012, 2115, 5156, 8756, 2609, 2008, 3499, 1005, 14676, 1005, 1997, 2309]...
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...

Text: So they should sue the newspaper I got it from for printing it. The article didn't say anything abou
  Tokens (28): ['[CLS]', 'so', 'they', 'should', 'sue', 'the', 'newspaper', 'i', 'got', 'it', 'from', 'for', 'printing', 'it', '.', 'the', 'article', 'didn', "'", 't']...
  input_ids: [101, 2061, 2027, 2323, 9790, 1996, 3780, 1045, 2288, 2009, 2013, 2005, 8021, 2009, 1012, 1996, 3720, 2134, 1005, 1056]...
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [20]:
# Пример padding и truncation
texts_pair = [
    "Short post about graphics.",
    "This is a much longer newsgroup post that discusses various aspects of computer graphics rendering, including ray tracing algorithms and texture mapping techniques used in modern 3D applications.",
]

encoded_padded = tokenizer(texts_pair, padding=True, truncation=True, max_length=32, return_tensors="pt")

for i, text in enumerate(texts_pair):
    tokens = tokenizer.convert_ids_to_tokens(encoded_padded["input_ids"][i])
    print(f"Text: {text}")
    print(f"  Tokens: {tokens}")
    print(f"  attention_mask: {encoded_padded['attention_mask'][i].tolist()}")
    pad_count = tokens.count("[PAD]")
    truncated = len(tokenizer.encode(text, add_special_tokens=True)) > 32
    print(f"  PAD tokens: {pad_count}, Truncated: {truncated}")
    print()

Text: Short post about graphics.
  Tokens: ['[CLS]', 'short', 'post', 'about', 'graphics', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  PAD tokens: 25, Truncated: False

Text: This is a much longer newsgroup post that discusses various aspects of computer graphics rendering, including ray tracing algorithms and texture mapping techniques used in modern 3D applications.
  Tokens: ['[CLS]', 'this', 'is', 'a', 'much', 'longer', 'news', '##group', 'post', 'that', 'discusses', 'various', 'aspects', 'of', 'computer', 'graphics', 'rendering', ',', 'including', 'ray', 'tracing', 'algorithms', 'and', 'texture', 'mapping', 'techniques', 'used', 'in', 'modern', '3d', 'applications', '[SEP]']

## 4. Инференс готовой pretrained модели

In [21]:
# Используем готовую модель для sentiment analysis — она НЕ обучена на тематическую классификацию
pretrained_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

inference_texts = [
    "The new OpenGL library provides better rendering performance for 3D graphics.",
    "The Yankees won the game 5-3 with a home run in the eighth inning.",
    "NASA announced a new mission to explore the moons of Jupiter.",
    "The government passed a controversial bill on immigration reform.",
    "What is the best graphics card for ray tracing in 2024?",
]

expected = ["comp.graphics", "rec.sport.baseball", "sci.space", "talk.politics.misc", "comp.graphics"]

print("Инференс готовой модели (sentiment-analysis, бинарная):")
print()
results = pretrained_pipe(inference_texts)
for text, result, exp in zip(inference_texts, results, expected):
    print(f"  Text: {text}")
    print(f"  Ожидаемая тема: {exp}")
    print(f"  Prediction: {result['label']} (score: {result['score']:.4f})")
    print()

HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 2s [Retry 2/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 4s [Retry 3/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 8s [Retry 4/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 8s [Retry 5/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Device set to use cp

Инференс готовой модели (sentiment-analysis, бинарная):

  Text: The new OpenGL library provides better rendering performance for 3D graphics.
  Ожидаемая тема: comp.graphics
  Prediction: POSITIVE (score: 0.9062)

  Text: The Yankees won the game 5-3 with a home run in the eighth inning.
  Ожидаемая тема: rec.sport.baseball
  Prediction: POSITIVE (score: 0.9992)

  Text: NASA announced a new mission to explore the moons of Jupiter.
  Ожидаемая тема: sci.space
  Prediction: POSITIVE (score: 0.9918)

  Text: The government passed a controversial bill on immigration reform.
  Ожидаемая тема: talk.politics.misc
  Prediction: NEGATIVE (score: 0.8481)

  Text: What is the best graphics card for ray tracing in 2024?
  Ожидаемая тема: comp.graphics
  Prediction: POSITIVE (score: 0.9994)



Готовая модель `distilbert-base-uncased-finetuned-sst-2-english` обучена на бинарный sentiment analysis (POSITIVE/NEGATIVE) и не может определить тематику сообщения (comp.graphics, rec.sport.baseball, sci.space, talk.politics.misc). Она лишь оценивает тональность текста, что не соответствует нашей задаче тематической классификации. Необходим fine-tuning.

## 5. Fine-tuning для классификации

In [22]:
MAX_LENGTH = 128

# Создаём torch Dataset
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, padding="max_length", truncation=True,
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

tok_train = NewsDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
tok_val = NewsDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
tok_test = NewsDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

print(f"Train:      {len(tok_train)}")
print(f"Validation: {len(tok_val)}")
print(f"Test:       {len(tok_test)}")

Train:      1903
Validation: 336
Test:       1490


In [24]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)
model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 66,956,548


In [25]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    compute_metrics=compute_metrics,
)

In [26]:
trainer.train()

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.479200,0.357921,0.883929,0.881857
2,0.253400,0.322269,0.877976,0.875816
3,0.167700,0.304202,0.886905,0.885387


/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=357, training_loss=0.3800263151067312, metrics={'train_runtime': 199.5654, 'train_samples_per_second': 28.607, 'train_steps_per_second': 1.789, 'total_flos': 189070838111232.0, 'train_loss': 0.3800263151067312, 'epoch': 3.0})

In [27]:
# Результаты на validation
val_results = trainer.evaluate()
print("Validation results:")
for k, v in val_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation results:
  eval_loss: 0.3042
  eval_accuracy: 0.8869
  eval_f1_macro: 0.8854
  eval_runtime: 2.8273
  eval_samples_per_second: 118.8400
  eval_steps_per_second: 3.8910
  epoch: 3.0000


## 6. Оценка на test и анализ ошибок

In [28]:
# Финальная оценка на test (один раз)
test_results = trainer.evaluate(tok_test)
print("Test results:")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test results:
  eval_loss: 0.3630
  eval_accuracy: 0.8812
  eval_f1_macro: 0.8794
  eval_runtime: 9.3752
  eval_samples_per_second: 158.9300
  eval_steps_per_second: 5.0130
  epoch: 3.0000


In [29]:
# Предсказания на test
test_predictions = trainer.predict(tok_test)
preds = np.argmax(test_predictions.predictions, axis=-1)
true_labels_arr = test_predictions.label_ids

test_accuracy = accuracy_score(true_labels_arr, preds)
test_f1 = f1_score(true_labels_arr, preds, average="macro")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test F1 macro: {test_f1:.4f}")
print()
print(classification_report(true_labels_arr, preds, target_names=label_names))

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test accuracy: 0.8812
Test F1 macro: 0.8794

                    precision    recall  f1-score   support

     comp.graphics       0.92      0.92      0.92       389
rec.sport.baseball       0.89      0.92      0.91       397
         sci.space       0.86      0.83      0.84       394
talk.politics.misc       0.86      0.85      0.85       310

          accuracy                           0.88      1490
         macro avg       0.88      0.88      0.88      1490
      weighted avg       0.88      0.88      0.88      1490



In [30]:
# Матрица ошибок
cm = confusion_matrix(true_labels_arr, preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Test)")
plt.tight_layout()
plt.savefig("artifacts/confusion_matrix.png", dpi=150)
plt.show()
print("Saved: artifacts/confusion_matrix.png")

Saved: artifacts/confusion_matrix.png


/var/folders/n_/mqpqp2w54_n0gnng_thc3sv40000gq/T/ipykernel_15753/2320629137.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
# Примеры предсказаний
confidences = torch.softmax(torch.tensor(test_predictions.predictions), dim=-1).max(dim=-1).values.numpy()

df_preds = pd.DataFrame({
    "text": [t.replace("\n", " ").strip()[:300] for t in test_texts],
    "true_label": [label_names[l] for l in true_labels_arr],
    "pred_label": [label_names[p] for p in preds],
    "confidence": np.round(confidences, 4),
})

# 10 правильных + 10 ошибок
correct = df_preds[df_preds["true_label"] == df_preds["pred_label"]].head(10)
errors = df_preds[df_preds["true_label"] != df_preds["pred_label"]].head(10)
sample_df = pd.concat([correct, errors]).reset_index(drop=True)
sample_df.to_csv("artifacts/sample_predictions.csv", index=False)
print("Saved: artifacts/sample_predictions.csv")
print(f"Total errors: {(df_preds['true_label'] != df_preds['pred_label']).sum()} / {len(df_preds)}")

Saved: artifacts/sample_predictions.csv
Total errors: 177 / 1490


In [32]:
# Примеры ошибок модели
print("Примеры ошибок модели:")
print("=" * 80)
error_df = df_preds[df_preds["true_label"] != df_preds["pred_label"]].head(10)
for _, row in error_df.iterrows():
    print(f"Text: {row['text'][:120]}...")
    print(f"  True: {row['true_label']} | Pred: {row['pred_label']} | Conf: {row['confidence']:.3f}")
    print()

Примеры ошибок модели:
Text: ...
  True: sci.space | Pred: rec.sport.baseball | Conf: 0.370

Text: You are forced everyday to associate with people that you do not wish to, and there isn't even a law that makes you do i...
  True: talk.politics.misc | Pred: rec.sport.baseball | Conf: 0.914

Text: Yes!  Just take money from the profitable commercial enterprises and give it to the government to "redistribute."  Gover...
  True: sci.space | Pred: talk.politics.misc | Conf: 0.881

Text: exit...
  True: sci.space | Pred: rec.sport.baseball | Conf: 0.308

Text: As does the idea that a CS gas canister can get hot enough to ignite dry baled hay....
  True: talk.politics.misc | Pred: sci.space | Conf: 0.927

Text: dead? I saw David Koresh at a local 7-11.........
  True: talk.politics.misc | Pred: rec.sport.baseball | Conf: 0.970

Text: Well ... Have a look at a new journal: Journal of Experimental Mathematics It has several Fields medallists on its edito...
  True: sci.space | Pred: comp.graph

## Краткий анализ ошибок

Типичные ошибки модели связаны с пересечением тематик:
- **sci.space vs talk.politics.misc**: обсуждения бюджета NASA или государственного финансирования космических программ попадают в обе категории.
- **comp.graphics vs sci.space**: визуализация космических данных и компьютерная графика в научных приложениях.
- **talk.politics.misc vs rec.sport.baseball**: политические аспекты спорта (скандалы, допинг).

Модель уверенно различает rec.sport.baseball и comp.graphics (наименее пересекающиеся темы), но испытывает трудности на текстах, лежащих на стыке тематик.